In [1]:
import pandas as pd
import numpy as np
import calendar
import glob

In [ ]:


# Cargar como NDJSON (una línea = un registro)
df = pd.read_json('bicimad_2022/202201.json', lines=True)

df.head()


,_id,stations
0,2022-01-01T00:13:20.603583,"[{'activate': 1, 'name': 'Puerta del Sol A', '..."
1,2022-01-01T01:13:21.911079,"[{'activate': 1, 'name': 'Puerta del Sol A', '..."
2,2022-01-01T02:13:23.718951,"[{'activate': 1, 'name': 'Puerta del Sol A', '..."
3,2022-01-01T03:13:23.902654,"[{'activate': 1, 'name': 'Puerta del Sol A', '..."
4,2022-01-01T04:13:26.826536,"[{'activate': 1, 'name': 'Puerta del Sol A', '..."


In [2]:
# Expandir la columna 'stations' (lista de dicts) en filas individuales
df_stations = df.explode('stations').reset_index(drop=True)

# Convertir los dicts de 'stations' en columnas separadas
df_stations = pd.concat(
    [df_stations['_id'], pd.json_normalize(df_stations['stations'])],
    axis=1
)

df_stations.head()


,_id,activate,name,reservations_count,light,total_bases,free_bases,number,longitude,no_available,address,latitude,dock_bikes,id
0,2022-01-01T00:13:20.603583,1,Puerta del Sol A,0,3,30,0,1a,-3.7018341,1,Puerta del Sol nº 1,40.4172137,0,1
1,2022-01-01T00:13:20.603583,1,Puerta del Sol B,0,3,30,0,1b,-3.701602938060457,1,Puerta del Sol nº 1,40.41731271011562,0,2
2,2022-01-01T00:13:20.603583,1,Miguel Moya,0,0,24,16,2,-3.7058415,0,Calle Miguel Moya nº 1,40.4205886,7,3
3,2022-01-01T00:13:20.603583,1,Plaza Conde Suchil,0,1,18,1,3,-3.7069171,0,Plaza del Conde del Valle de Súchil nº 3,40.4302937,14,4
4,2022-01-01T00:13:20.603583,1,Malasaña,0,1,24,2,4,-3.7025875,0,Calle Manuela Malasaña nº 5,40.4285524,17,5


### Limpieza de datos basica

Vamos a analizar cada una de las columnas que tenemos en nuestro JSON:

# Columnas eliminadas del JSON

- **`activate`** → siempre vale 1, no aporta información.
- **`light`** → es un resumen de `dock_bikes`/`free_bases` que ya tenemos, y usarla sería fuga de información.
- **`reservations_count`** → casi siempre 0 y es una señal de minutos, no sirve para predecir a un día.
- **`no_available`** → solo la usamos para filtrar estaciones caídas y luego se descarta.
- **`number`** → código de estación redundante, ya unimos con `id`.
- **`address`** → texto redundante, ya tenemos `id`, `name` y coordenadas.

In [3]:
df_stations = df_stations[df_stations["no_available"] == 0]

In [ ]:
df_stations = df_stations.drop(columns=['activate', 'light', 'reservations_count', 'no_available', 'number', 'address'])
df_stations.head()

Convertimos las cordenadas a números y añadimos una nueva columna calculando el porcentaje de ocupacion de la estación

In [7]:
# 1) Coordenadas de texto a número
df_stations["longitude"] = df_stations["longitude"].astype(float)
df_stations["latitude"]  = df_stations["latitude"].astype(float)

# 2) Columna de ocupación (denominador = anclajes que de verdad funcionan)
df_stations["ocupation"] = df_stations["dock_bikes"] / (df_stations["dock_bikes"] + df_stations["free_bases"])
df_stations.head()

,_id,name,total_bases,free_bases,longitude,latitude,dock_bikes,id,ocupation
2,2022-01-01T00:13:20.603583,Miguel Moya,24,16,-3.705842,40.420589,7,3,0.304348
3,2022-01-01T00:13:20.603583,Plaza Conde Suchil,18,1,-3.706917,40.430294,14,4,0.933333
4,2022-01-01T00:13:20.603583,Malasaña,24,2,-3.702587,40.428552,17,5,0.894737
5,2022-01-01T00:13:20.603583,Fuencarral,27,14,-3.702050,40.428520,10,6,0.416667
6,2022-01-01T00:13:20.603583,Colegio Arquitectos,24,11,-3.698447,40.424148,10,7,0.476190


In [8]:
def procesar_mes(ruta_json):
    # Lectura defensiva: prueba UTF-8 y si falla, latin-1
    try:
        df = pd.read_json(ruta_json, lines=True)
    except UnicodeDecodeError:
        print(f"  ⚠ Codificación no-UTF8 detectada, leyendo como latin-1")
        df = pd.read_json(ruta_json, lines=True, encoding="latin-1")

    df_stations = df.explode('stations').reset_index(drop=True)
    df_stations = pd.concat(
        [df_stations['_id'], pd.json_normalize(df_stations['stations'])],
        axis=1
    )
    df_stations = df_stations[df_stations['no_available'] == 0]
    df_stations = df_stations.drop(columns=['activate', 'light', 'reservations_count', 'no_available', 'number', 'address'])
    df_stations["longitude"] = df_stations["longitude"].astype(float)
    df_stations["latitude"]  = df_stations["latitude"].astype(float)
    suma = df_stations["dock_bikes"] + df_stations["free_bases"]
    df_stations["ocupation"] = np.where(suma > 0, df_stations["dock_bikes"] / suma, np.nan)
    return df_stations

In [ ]:
def comprobaciones(df):
    print(f"Filas: {len(df):,}  |  Columnas: {list(df.columns)}\n")

    # 1) Estaciones
    print(f"Estaciones únicas (id): {df['id'].nunique()}  (esperado ~264)")

    # 2) Cobertura temporal
    ts = pd.to_datetime(df['_id'])
    dias_mes = calendar.monthrange(ts.dt.year.iloc[0], ts.dt.month.iloc[0])[1]
    esperadas = dias_mes * 24
    print(f"Fotos horarias: {df['_id'].nunique()}  (esperado {esperadas})")
    print(f"Rango: {ts.min()}  ->  {ts.max()}\n")

    # 3) Ocupación: rango, valores imposibles y NaN
    oc = df['ocupation']
    fuera = ((oc < 0) | (oc > 1)).sum() 
    print(f"Ocupación -> min: {oc.min():.3f} | media: {oc.mean():.3f} | max: {oc.max():.3f}")
    print(f"Valores fuera de [0,1]: {fuera}  (debe ser 0)")
    print(f"NaN en ocupación: {oc.isna().sum()}  (debe ser 0 tras filtrar caídas)\n")

    # 4) Duplicados (misma estación en la misma foto)
    dups = df.duplicated(subset=['id', '_id']).sum()
    print(f"Filas duplicadas (id + foto): {dups}  (debe ser 0)")

In [18]:
df = procesar_mes("bicimad_2022_json/202210.json")
comprobaciones(df)

Filas: 187,119  |  Columnas: ['_id', 'name', 'total_bases', 'free_bases', 'longitude', 'latitude', 'dock_bikes', 'id', 'ocupation']

Estaciones únicas (id): 258  (esperado ~264)
Fotos horarias: 738  (esperado 744)
Rango: 2022-10-01 01:05:08.526286  ->  2022-10-31 23:19:18.083498

Ocupación -> min: 0.000 | media: 0.404 | max: 1.000
Valores fuera de [0,1]: 0  (debe ser 0)
NaN en ocupación: 56  (debe ser 0 tras filtrar caídas)

Filas duplicadas (id + foto): 0  (debe ser 0)


Ahora hacemos lo mismo para todos los meses de 2023

In [19]:
tablas = []
for ruta in sorted(glob.glob("bicimad_2022_json/2022*.json")):
    print(f"\n===== {ruta} =====")
    df_mes = procesar_mes(ruta)
    comprobaciones(df_mes)
    tablas.append(df_mes)

df_anual = pd.concat(tablas, ignore_index=True)
df_anual.to_parquet("ocupacion_2022.parquet", index=False)
print(f"\nDataset final: {len(df_anual):,} filas guardadas en ocupacion_2022.parquet")


===== bicimad_2022_json\202201.json =====
Filas: 190,784  |  Columnas: ['_id', 'name', 'total_bases', 'free_bases', 'longitude', 'latitude', 'dock_bikes', 'id', 'ocupation']

Estaciones únicas (id): 259  (esperado ~264)
Fotos horarias: 740  (esperado 744)
Rango: 2022-01-01 00:13:20.603583  ->  2022-01-31 23:35:11.044151

Ocupación -> min: 0.000 | media: 0.475 | max: 1.000
Valores fuera de [0,1]: 0  (debe ser 0)
NaN en ocupación: 5  (debe ser 0 tras filtrar caídas)

Filas duplicadas (id + foto): 0  (debe ser 0)

===== bicimad_2022_json\202202.json =====
Filas: 173,075  |  Columnas: ['_id', 'name', 'total_bases', 'free_bases', 'longitude', 'latitude', 'dock_bikes', 'id', 'ocupation']

Estaciones únicas (id): 260  (esperado ~264)
Fotos horarias: 668  (esperado 672)
Rango: 2022-02-01 00:35:12.131944  ->  2022-02-28 23:54:20.406908

Ocupación -> min: 0.000 | media: 0.475 | max: 1.000
Valores fuera de [0,1]: 0  (debe ser 0)
NaN en ocupación: 35  (debe ser 0 tras filtrar caídas)

Filas dupli